In [97]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [98]:
# 1 Load datset
raw_data = pd.read_csv("raw_data.csv")

In [99]:
raw = raw_data.drop(["EmployeeCount","StandardHours", "Over18", "EmployeeNumber"],axis=1).copy()

In [100]:
raw['Attrition'] = raw['Attrition'].map({'No': 0, 'Yes': 1})  # 1 : Left, 0 : still there
raw['OverTime'] = raw['OverTime'].map({'No': 0, 'Yes': 1}) 

In [101]:
raw["TenureRatio"] = raw["YearsAtCompany"] / raw["TotalWorkingYears"]

In [102]:
raw['TenureRatio'] = raw['TenureRatio'].fillna(0)


In [103]:
raw["PromotionGap"] = raw["YearsAtCompany"] - raw["YearsSinceLastPromotion"]  

In [104]:
raw["IncomeLevelRatio"] = raw["MonthlyIncome"] / raw["JobLevel"]

In [105]:
X = raw.drop("Attrition", axis=1)
y = raw["Attrition"]


In [106]:
#Split the data

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(X, y):
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]


In [107]:
data_numeric = X_train.drop(["BusinessTravel","Department","EducationField","Gender","JobRole","MaritalStatus"],axis=1)
data_category = X_train[["BusinessTravel","Department","EducationField","Gender","JobRole","MaritalStatus"]]

# Pipelines
# Numerical pipeline

num_pipeline = Pipeline([
("imputer", SimpleImputer(strategy="median")),
("scaler", StandardScaler()),
])


In [108]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [109]:
num_cols = data_numeric.columns.tolist()
cat_cols = data_category.columns.tolist()

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [110]:
X_train_prepared = full_pipeline.fit_transform(X_train)
X_test_prepared = full_pipeline.transform(X_test)

In [113]:
X_train_prepared.shape

(1176, 53)

In [114]:
X_test_prepared.shape

(294, 53)